# Single-Track Library Cartoon

Generates a compact single-track matplotlib cartoon showing exon structure with library amplicon crosshatching.
Protein domains (if provided) are shown as colored overlays on CDS blocks. Variant stats are printed
to the right when a scores file is supplied.

**Required data:** Exon coordinates are fetched from the Ensembl REST API (requires internet access).
A targets TSV file with `editstart`/`editstop` columns is required for the library amplicon crosshatch.

**Alternative data source:** If you have a cartoon metadata Excel file (`exon_coords`, `metadata`, `lib_coords` sheets), you can load from that instead — see the commented block in the data loading cell.

**Required packages:** `sge-utils` (SimpleSGEViz) installed in the active environment.

**Saving:** PNG and SVG are supported. HTML output is not available for this matplotlib figure.

In [ ]:
import pandas as pd
from pathlib import Path
from sgeviz import io
from sgeviz.figures import single_track_cartoon

In [ ]:
# --- Configuration ---
gene = "GENE"                   # gene symbol (used in Ensembl lookup and figure label)
transcript_id = None            # Ensembl transcript ID (e.g. 'ENST00000357654'); None = auto-select canonical
assembly = "GRCh38"             # genome assembly: 'GRCh38' or 'GRCh37'
targets_path = "targets.tsv"   # required - TSV with editstart/editstop columns
excel_path = None               # optional - pipeline output Excel file for variant stats sidebar
domains_path = None             # optional - domain annotation file (CSV or Excel)
output_path = f"{gene}_single_track_cartoon.png"  # PNG or SVG only (HTML not supported)

# --- Plot customization (optional) ---
fig_width = 10          # figure width in inches
exon_color = "#d0d0d0"  # exon block fill color (domains override this where applicable)

In [ ]:
# --- Load exon coordinates ---
# Option A (default): fetch from Ensembl REST API
exon_df, _, meta_df = io.fetch_exon_coords(gene, transcript_id=transcript_id, assembly=assembly)

# Option B: load from a cartoon metadata Excel file instead
# cartoon_path = "GENE_cartoon.xlsx"
# xl = pd.ExcelFile(cartoon_path)
# exon_df = xl.parse("exon_coords")
# meta_df = xl.parse("metadata")
# lib_df_cartoon = xl.parse("lib_coords") if "lib_coords" in xl.sheet_names else None

# --- Load library amplicon targets (required) ---
if targets_path is None:
    raise ValueError(
        "targets_path must be set. Provide a TSV file with editstart/editstop columns."
    )
targets_df = pd.read_csv(targets_path, sep="\t")
lib_df = targets_df[["editstart", "editstop"]].rename(
    columns={"editstart": "start", "editstop": "end"}
)
print(f"Loaded {len(lib_df)} library amplicons")

# --- Optionally load scores for variant stats sidebar ---
scores_df = None
if excel_path is not None:
    scores_df = pd.read_excel(excel_path, sheet_name="scores")
    print(f"Loaded {len(scores_df)} variants for stats sidebar")

In [ ]:
# --- Generate plot ---
domains = Path(domains_path) if domains_path else None
fig = single_track_cartoon.make_plot(
    gene, exon_df, meta_df, lib_df,
    scores_df=scores_df,
    domains_path=domains,
    exon_color=exon_color,
    fig_width=fig_width,
)
fig

In [ ]:
# --- Save ---
fig.savefig(output_path, bbox_inches="tight", dpi=150)
print(f"Saved: {output_path}")